# 第4章 ローカルLLMの活用ワークフロー 実践

この Notebook では、ブラウザ Chat UI、Python API、Agentic AI、レビューを1つの作業の流れとして体験します。読むだけでなく、実際に画面を開いて、どの入口が何に向いているかを確かめます。

In [ ]:
from pathlib import Path
import os
import sys

# Notebook をどこから開いても helper を import できるようにします。
search_roots = [Path.cwd()]
env_root = os.environ.get("LOCAL_LLM_REPO_ROOT")
if env_root:
    search_roots.append(Path(env_root))
search_roots.append(Path("C:/LLM"))

seen = set()
for root in search_roots:
    current = root.resolve()
    for candidate in [current, *current.parents]:
        if candidate in seen:
            continue
        seen.add(candidate)
        helper_dir = candidate / "notebooks"
        if (helper_dir / "local_llm_practice.py").exists():
            sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise RuntimeError(
        "notebooks/local_llm_practice.py が見つかりません。"
        "C:/LLM か notebooks/ 配下で開くか、LOCAL_LLM_REPO_ROOT を設定してください。"
    )

from local_llm_practice import (
    configure_local_caches,
    copy_to_clipboard,
    load_chapter,
    ollama_generate,
    open_aider_terminal,
    prepare_aider_practice_workspace,
    print_headings,
    REPO_ROOT,
    DOCS_DIR,
    WORK_DIR,
    start_ollama_chat_ui,
    stop_ollama_chat_ui,
)

configure_local_caches()
print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR :", DOCS_DIR)
print("WORK_DIR :", WORK_DIR)

chapter_path, chapter_text = load_chapter("04-tool-use.md")
print(chapter_path)
print_headings(chapter_text)


## 1. Chat UI で相談を始める

次のセルでブラウザ Chat UI を開きます。開いたら、次を試します。

1. `文書作成、コーディング、資料確認でローカルLLMを今日から使う小さな用途を5つ挙げてください。` と入力する
2. 返答を見て、1つ選ぶ
3. `その用途を今日試す手順にしてください。` と追加で頼む

Chat UI は、会話しながら作業を具体化する入口です。

In [ ]:
initial_prompt = "文書作成、コーディング、資料確認でローカルLLMを今日から使う小さな用途を5つ挙げてください。"
start_ollama_chat_ui()
copy_to_clipboard(initial_prompt)


## 2. Python API で同じ相談を再現する

Chat UI で選んだ用途を `selected_use` に書き、APIで手順化します。Python API は、同じ依頼を保存して再実行する時に向いています。

In [ ]:
selected_use = "手順書をもとに利用者向けFAQの下書きを作る"
api_workflow_prompt = f"""
次の用途を、今日試せる手順にしてください。
用途: {selected_use}
条件:
- 30分以内で試せる
- Chat UI、Python API、Agentic AI のどれを使うか明記する
- 最後に人間が確認すべきことを書く
""".strip()

print(ollama_generate(api_workflow_prompt, temperature=0.2))

## 3. Agentic AI を起動して、編集なしの調査を体験する

次のセルで aider を別PowerShellで起動します。開いた PowerShell で次を入力します。

1. `/help`
2. `/add README.md`
3. `/add notebooks/local-llm-customization/00-index.ipynb`
4. `この教材で初心者が最初にたどる導線を説明してください。まだ編集しないでください。`
5. 返答を確認する
6. `/exit`

Agentic AI は、ファイルやリポジトリ文脈を読ませて相談できる入口です。最初の体験では編集しません。
実リポジトリを誤って編集しないように、この章では `work/aider-practice/` に作る練習用 workspace を開きます。ここは `.gitignore` されているため、教材の tracked files は変更されません。


In [ ]:
practice_path = prepare_aider_practice_workspace()
open_aider_terminal(practice_path)


## 4. レビューとして使う体験

最後に、Chat UI または Agentic AI にレビュー役を頼みます。ここでは新しい作業を任せるのではなく、すでに出た答えを点検させます。

試すこと:

1. Chat UI か aider のどちらかを開く
2. 自分が作った手順を貼る
3. `初心者が実行できるか、危ない操作が含まれていないか、確認不足がないかをレビューしてください。重大な点から順に挙げてください。` と頼む
4. 指摘を読んで、必要なら手順を直す

レビュー用途では、LLMの答えをそのまま採用せず、人間が最後に確認します。

## 結果の読み方 / 次へ進む判断

- Chat UI で作業案を広げ、Python API で同じ依頼を保存して再実行できれば、入口ごとの役割を確認できています。
- aider ではファイルを読ませた調査だけを行い、編集しない依頼で止められれば、安全な agentic coding の最初の型を体験できています。
- レビュー依頼で重大な確認点から返ってくれば、LLM を「作業者」だけでなく「点検者」として使う入口を確認できています。

この先は、通常利用で残る不足が本当に学習で解決するものかを考えてから、第5章の LoRA 実学習へ進みます。

Chat UI を使い終わったら、同じ kernel で `stop_ollama_chat_ui()` を実行すると教材用サーバーを終了できます。Notebook の kernel を再起動しても終了します。
